In [1]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
import pytz

# === Time Setup ===
eastern = pytz.timezone("US/Eastern")
yesterday = datetime.now(eastern) - timedelta(days=1)
date_str = yesterday.strftime('%Y-%m-%d')

# === File Output Setup ===
output_dir = os.path.abspath(os.path.join("..", "data", "game-scores"))
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f"game_scores_{date_str}.csv")

# === Fetch Scores Function ===
def fetch_mlb_scores(target_date):
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={target_date.strftime("%Y-%m-%d")}'
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"❌ Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    game_data = []
    for game in games:
        home_team = game['teams']['home']['team']['name']
        away_team = game['teams']['away']['team']['name']

        home_score = game['teams']['home'].get('score', None)
        away_score = game['teams']['away'].get('score', None)

        game_info = {
            'Date': target_date.strftime('%Y-%m-%d'),
            'Home Team': home_team,
            'Home Score': home_score,
            'Away Team': away_team,
            'Away Score': away_score,
            'Status': game['status']['detailedState']
        }
        game_data.append(game_info)

    return pd.DataFrame(game_data)

# === Run ===
scores_df = fetch_mlb_scores(yesterday)

if scores_df is not None and not scores_df.empty:
    scores_df.to_csv(output_file, index=False)
    print(f"✅ Saved scores to {output_file}")
    print(scores_df)
else:
    print("⚠️ No scores found or failed to retrieve data.")


✅ Saved scores to /workspaces/MLB_Model/data/game-scores/game_scores_2025-05-30.csv
          Date              Home Team  Home Score             Away Team  \
0   2025-05-30           Chicago Cubs           2       Cincinnati Reds   
1   2025-05-30      Baltimore Orioles           2     Chicago White Sox   
2   2025-05-30  Philadelphia Phillies           2     Milwaukee Brewers   
3   2025-05-30      Toronto Blue Jays          11             Athletics   
4   2025-05-30    Cleveland Guardians           1    Los Angeles Angels   
5   2025-05-30          New York Mets           4      Colorado Rockies   
6   2025-05-30          Miami Marlins           0  San Francisco Giants   
7   2025-05-30         Atlanta Braves           1        Boston Red Sox   
8   2025-05-30          Texas Rangers          11   St. Louis Cardinals   
9   2025-05-30     Kansas City Royals           5        Detroit Tigers   
10  2025-05-30         Houston Astros           2        Tampa Bay Rays   
11  2025-05-30  